# Cheap-Talk Benchmark - Kaggle runner

One notebook for every campaign: pick a model, a session and a topology, run
it. All the logic lives in `campaign.py` in the repo, so fixes arrive with a
`git pull` and these cells never need editing.

> **Why it is built this way.** Kaggle executes the cells saved in *your*
> workspace, never the notebook stored in the repo. Logic kept in cells can
> only be fixed by hand-editing cells, and a stale broken cell will happily
> re-run after a pull. Likewise `!python ... $VAR` silently expands an
> undefined variable to an empty string instead of failing, so the run cell
> passes an argument list to `subprocess` instead.

**Before running**
1. Settings -> Accelerator -> `GPU T4 x2`
2. Settings -> Internet -> `On`
3. Add-ons -> Secrets -> add `HF_TOKEN` (needed for the gated Gemma and Llama
   models; not needed for Qwen)
4. For anything over ~2 h use **Save Version -> Save & Run All**, so a browser
   disconnect cannot kill the session.


## 1. Environment

Kaggle preinstalls torch and transformers; reinstalling them breaks the kernel.
Only `bitsandbytes` and `python-dotenv` are missing.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch, transformers
assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:  ", torch.cuda.get_device_name(0), f"({gb:.1f} GB)")
print("torch:", torch.__version__, " transformers:", transformers.__version__)


## 2. Code

Clones on the first run, fast-forwards on every later one, so a session always
picks up the newest fixes.


In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"
REPO_DIR = "/kaggle/working/repo"

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
# os.chdir rather than the %cd magic: no variable expansion involved, and it
# raises if the path is wrong instead of quietly landing somewhere else.
os.chdir(REPO_DIR)
head = subprocess.run(["git", "log", "--oneline", "-1"],
                      capture_output=True, text=True).stdout
print("cwd :", os.getcwd())
print("HEAD:", head.strip())


## 3. Credentials

Only needed for gated models (Gemma, Llama). Accept the licence on the model's
HuggingFace page first, or the download 403s.


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HUGGINGFACE_API_KEY"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen models):", e)


## 4. Choose the run

| Model | Session A | Session B | Total | Sessions needed |
|---|---|---|---|---|
| `google/gemma-2-2b-it` | ~1.6 h | ~1.3 h | ~2.9 h | 2 |
| `Qwen/Qwen3-4B` | ~2.5 h | ~2.1 h | ~4.6 h | 1 (use `ALL`) |
| `Qwen/Qwen2.5-7B-Instruct` | ~4.3 h | ~3.7 h | ~8.0 h | 2 |
| `meta-llama/Llama-3.1-8B-Instruct` | ~4.6 h | ~3.9 h | ~8.5 h | 2 |
| `google/gemma-2-9b-it` | ~5.2 h | ~4.5 h | ~9.7 h | 2 |

**Session A** = baseline + no_sense + silence + counterfactual (50 runs).
**Session B** = the three framings (30 runs). **ALL** = both (80 runs).
Estimates are +/-30% on a T4; the run prints its real wall time at the end.

`max_new_tokens` is not set here: `campaign.py` looks up each model's value
from the star campaign so a star-vs-cycle comparison carries no extra
confound.

**Outstanding jobs**, in order:

1. **RQ4, the filtered cell.** `framing_competitive` on the star, PD only, with
   `MESSAGE_FILTER = "F3_relative_gain"`, for all five models. 5 run files and
   about 640 LLM calls each, roughly 1.5-2 h per model. `max_new_tokens` comes
   from the session-B table automatically, so the filtered run matches its
   unfiltered twin and the comparison carries no extra confound.

   What to read off it: first the realised block rate from `messages_blocked`
   (55.5% offline -- if it moves in the loop, that is itself a result), then
   cooperation against the three points already known, `framing_competitive`
   -0.12, `silence` +0.19, and the unfiltered baseline.

2. Optional afterwards: the same cell on SH, and `F1_competitive` as a
   near-`silence` control.


In [ ]:
MODEL    = "Qwen/Qwen3-4B"
# MODEL  = "google/gemma-2-9b-it"
# MODEL  = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL  = "Qwen/Qwen2.5-7B-Instruct"
# MODEL  = "google/gemma-2-2b-it"

SESSION  = "B"        # "A", "B", "C", or "ALL"
TOPOLOGY = "star"     # "cycle", "star", "clique", "line"

# Run only part of a session. None = the whole session. Use it when a session
# does not fit in the 12 h limit, or to resume one that was killed part-way:
# list the scenarios that have not been produced yet.
SCENARIOS = ["framing_competitive"]

# Restrict the games. None = both, which is what every campaign so far did.
GAMES = ["pd"]

# RQ4: channel-side moderation. A message that trips the filter is not
# delivered -- the sender writes it, is never told, and the neighbours receive
# an empty string, exactly what the `silence` scenario delivers.
#   "none"              every earlier run, unchanged
#   "F3_relative_gain"  blocks 55% of framing_competitive, 0% of baseline
#   "F1_competitive"    blocks 93% -- close to `silence`, kept as a control
# Filtered output lands in its own directory tree, so it can never be ingested
# as though it were the unfiltered cell.
MESSAGE_FILTER = "F3_relative_gain"


## 5. Preview

Costs nothing and loads no model. Check the numbers before spending GPU hours:
a wrong model id or session shows up here instead of ten minutes into a load.


In [ ]:
import subprocess, sys

# Kaggle runs the cells saved in YOUR workspace, not the repo copy. A stale run
# cell cannot drop an argument silently: if the config cell above is the old
# version, this raises instead of quietly running the wrong experiment.
for _v in ("MODEL", "SESSION", "TOPOLOGY", "SCENARIOS", "GAMES", "MESSAGE_FILTER"):
    if _v not in globals():
        raise NameError(
            f"{_v} is not defined. The config cell above is the old version -- "
            "update it before running, or this cell silently drops the argument."
        )

ARGS = ["--model", MODEL, "--session", SESSION, "--topology", TOPOLOGY]
if SCENARIOS:
    ARGS += ["--scenarios", *SCENARIOS]
if GAMES:
    ARGS += ["--games", *GAMES]
if MESSAGE_FILTER != "none":
    ARGS += ["--message-filter", MESSAGE_FILTER]
print("campaign.py", *ARGS)

subprocess.run([sys.executable, "campaign.py", *ARGS, "--dry-run"], check=True)


## 6. Run

`campaign.py` runs the sweep, then verifies what was actually produced - the
number of run files and the topology recorded inside them - and only packages
a zip if that checks out. It exits 2 when the sweep fails and 3 when the output
does not verify; `check=True` turns either into a failed cell, so a broken
session can never masquerade as a good one.


In [ ]:
import subprocess, sys

# Kaggle runs the cells saved in YOUR workspace, not the repo copy. A stale run
# cell cannot drop an argument silently: if the config cell above is the old
# version, this raises instead of quietly running the wrong experiment.
for _v in ("MODEL", "SESSION", "TOPOLOGY", "SCENARIOS", "GAMES", "MESSAGE_FILTER"):
    if _v not in globals():
        raise NameError(
            f"{_v} is not defined. The config cell above is the old version -- "
            "update it before running, or this cell silently drops the argument."
        )

ARGS = ["--model", MODEL, "--session", SESSION, "--topology", TOPOLOGY]
if SCENARIOS:
    ARGS += ["--scenarios", *SCENARIOS]
if GAMES:
    ARGS += ["--games", *GAMES]
if MESSAGE_FILTER != "none":
    ARGS += ["--message-filter", MESSAGE_FILTER]
print("campaign.py", *ARGS)

subprocess.run([sys.executable, "campaign.py", *ARGS], check=True)


## 7. Afterwards

The zip is written to `/kaggle/working/` and appears in the Output panel.

1. Download it into `diplomatikh/drive_sync/` and upload it to the Drive folder
   `cheaptalk_bench_results`.
2. Append the runs to the `Runs` sheet of `cheaptalk_results_tracker.xlsx`;
   the `Combinations` and `Results` sheets recompute themselves.
3. Add a row to `TRACK_RECORD.md` with the wall-clock time printed above, so
   the remaining estimates can be recalibrated.

If the run dies midway, the per-scenario zips already written to
`/kaggle/working/` are still good - grab them before closing the session.
